# 07 - Final 12-Month Forecast

> Retrain the best model (Prophet) on the **full historical dataset** (2011-2014) and generate the official 12-month sales forecast for 2015.

**Output**: `predictions/final_12_month_forecast.csv`

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib, warnings
from prophet import Prophet
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
COLORS = {'primary':'#003B73','secondary':'#0074B7','success':'#27AE60','alert':'#BF212F'}
print("Libraries loaded")

## 1. Load Full Monthly Series

In [ ]:
monthly = pd.read_csv('../data/monthly_sales.csv', parse_dates=['ds'])
metrics = pd.read_csv('../evaluation/model_comparison.csv')
best = metrics.iloc[0]
print(f"Best Model: {best['Model']} (MAPE: {best['MAPE (%)']:.2f}%, R2: {best['R2']:.4f})")
print(f"Training on full dataset: {len(monthly)} months ({monthly['ds'].min().date()} to {monthly['ds'].max().date()})")

## 2. Retrain Prophet on Full Data

In [ ]:
final_model = Prophet(yearly_seasonality=True, weekly_seasonality=False, daily_seasonality=False,
                      seasonality_mode='multiplicative')
final_model.fit(monthly)
print("Final Prophet model trained on all 48 months")

## 3. Generate 12-Month Forecast (2015)

In [ ]:
future = final_model.make_future_dataframe(periods=12, freq='MS')
forecast = final_model.predict(future)

# Extract the 2015 forecast
forecast_2015 = forecast[forecast['ds'] >= '2015-01-01'][['ds','yhat','yhat_lower','yhat_upper']].copy()
forecast_2015.rename(columns={'yhat':'forecast','yhat_lower':'forecast_lower','yhat_upper':'forecast_upper'}, inplace=True)
forecast_2015['forecast'] = forecast_2015['forecast'].round(2)
forecast_2015['forecast_lower'] = forecast_2015['forecast_lower'].round(2)
forecast_2015['forecast_upper'] = forecast_2015['forecast_upper'].round(2)

print("12-Month Sales Forecast for 2015:")
print("=" * 65)
for _, row in forecast_2015.iterrows():
    print(f"  {row['ds'].strftime('%B %Y'):>15}: ${row['forecast']:>12,.2f}  (${row['forecast_lower']:>12,.2f} - ${row['forecast_upper']:>12,.2f})")
print("=" * 65)
print(f"  {'TOTAL 2015':>15}: ${forecast_2015['forecast'].sum():>12,.2f}")

## 4. Visualize Final Forecast

In [ ]:
fig, ax = plt.subplots(figsize=(16, 7))

# Historical actuals
ax.plot(monthly['ds'], monthly['y'], color='black', linewidth=2, marker='o', markersize=4, label='Historical Sales (2011-2014)')

# 2015 forecast
ax.plot(forecast_2015['ds'], forecast_2015['forecast'], color=COLORS['primary'], linewidth=2.5, marker='D', markersize=6, linestyle='--', label='Prophet Forecast (2015)')
ax.fill_between(forecast_2015['ds'], forecast_2015['forecast_lower'], forecast_2015['forecast_upper'],
                color=COLORS['secondary'], alpha=0.2, label='95% Confidence Interval')

# Vertical separator
ax.axvline(pd.Timestamp('2015-01-01'), color='red', linestyle=':', alpha=0.6, linewidth=1.5)
ax.text(pd.Timestamp('2015-01-01'), ax.get_ylim()[1]*0.95, '  Forecast Start', color='red', fontsize=10)

ax.set_title('AI-Powered 12-Month Sales Forecast (2015)', fontsize=16, fontweight='bold')
ax.set_xlabel('Date', fontsize=12)
ax.set_ylabel('Monthly Sales ($)', fontsize=12)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x,_: f'${x/1000:.0f}K'))
ax.legend(loc='upper left', fontsize=11)
plt.tight_layout()
plt.savefig('../visualizations/final_forecast.png', bbox_inches='tight', dpi=150)
plt.show()

## 5. Save Final Outputs

In [ ]:
# Save the 12-month forecast
forecast_2015.to_csv('../predictions/final_12_month_forecast.csv', index=False)

# Save the retrained final model
joblib.dump(final_model, '../models/prophet_model_final.pkl')

# Also save the full historical + forecast combined for Streamlit
full = forecast[['ds','yhat','yhat_lower','yhat_upper']].merge(monthly, on='ds', how='left')
full.rename(columns={'y':'actual','yhat':'forecast','yhat_lower':'forecast_lower','yhat_upper':'forecast_upper'}, inplace=True)
full.to_csv('../predictions/full_forecast_with_actuals.csv', index=False)

print("Saved:")
print("  predictions/final_12_month_forecast.csv  - 12-month 2015 forecast")
print("  predictions/full_forecast_with_actuals.csv - Combined actuals + forecast")
print("  models/prophet_model_final.pkl           - Final retrained model")

## 6. Executive Forecast Summary

In [ ]:
total_2014 = monthly[monthly['ds'].dt.year == 2014]['y'].sum()
total_2015 = forecast_2015['forecast'].sum()
growth = ((total_2015 - total_2014) / total_2014) * 100

print("=" * 60)
print("EXECUTIVE FORECAST SUMMARY")
print("=" * 60)
print(f"Best Model:          Prophet (Multiplicative Seasonality)")
print(f"Validation MAPE:     {best['MAPE (%)']:.2f}% (Target: <15%)")
print(f"Validation R2:       {best['R2']:.4f} (Target: >0.80)")
print(f"2014 Actual Sales:   ${total_2014:,.2f}")
print(f"2015 Forecast Sales: ${total_2015:,.2f}")
print(f"Projected Growth:    {growth:+.2f}%")
print(f"Peak Month:          {forecast_2015.loc[forecast_2015['forecast'].idxmax(), 'ds'].strftime('%B %Y')}")
print("=" * 60)
print("\nPhase 6 (Machine Learning) COMPLETE.")
print("Proceed to Phase 7: 07_Streamlit_App/")